In [1]:
from langchain_community.document_loaders import YoutubeLoader
from langchain_core.documents import Document 
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api.proxies import WebshareProxyConfig
import concurrent.futures
import re
import json
import jsonlines
import os
import ipynb

/home/nando/anaconda3/envs/ds_cont/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
!pip install ipynb

In [3]:
from ipynb.fs.defs.download_vid import download_video
from ipynb.fs.defs.test_mp3 import transcribe

In [ ]:
# download -> transcribe -> delete the file -> save trans as langchain document
def video_procesing(video_url, 
                    ydl_opts = {
                                    'quiet': True,
                                    'noplaylist': True,
                                },
                    delete_file=True):

    # -> DOWNLOAD
    # standar_video_path = os.path.join("audio", "%(title)s.%(ext)s")
    # standar_video_path = os.path.join("audio", "temp")
    standar_video_path = "temp"

    downloading_path = download_video(url=video_url, video_path=standar_video_path)

    # -> TRANSCRIBE
    transcript = transcribe(video_path=downloading_path, model_name="tiny")

    # -> DELELETE FILE

    try:    
        if delete_file and os.path.exists(downloading_path):
            os.remove(downloading_path)
    except:

        print("The file does not exist") 

    # -> SAVE AS LANGCHAIN DOCUMENT
    document = Document(page_content = transcript, metadata = {"source": video_url})

    # ----> Cargar metadatos del video
    import yt_dlp

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info_dict = ydl.extract_info(video_url, download=False)
        
        title = info_dict.get('title', '')
        description = info_dict.get('description', '')

    # ----> Save the file
    document.page_content = f"{title} \n {description} \n {document.page_content}"
    document.metadata['title'] = title
    document.metadata['description'] = description
    return document

In [5]:
def save_to_doc(file_path, list_documents):
    print(f"Saving documents to {file_path}...")
    with jsonlines.open(file_path, mode='w') as writer:
        for doc in list_documents:
            # Convert the Document object to a dictionary
            writer.write(doc.dict())
    print("Save complete.")

# Main

In [ ]:
import time


docs = []
from urls import list_urls

for link_url in list_urls:
    try:
        start = time.time()
        document = video_procesing(link_url)
        end = time.time()

        print()
        print(f"Total time: {end-start}")
        print()
        docs.append(document)
    except:
        error = f"Error in: {link_url}"
        raise RuntimeError(error)

file_path = "Doc.jsonl"
save_to_doc(file_path, docs)
print()
print("="*40)
print("EVERYTHING WENT WELL :). Finished")
print("="*40)


[youtube] Extracting URL: https://www.youtube.com/watch?v=ixcgyeKnBJg
[youtube] ixcgyeKnBJg: Downloading webpage
[youtube] ixcgyeKnBJg: Downloading android vr player API JSON
[youtube] ixcgyeKnBJg: Downloading web player API JSON
[youtube] ixcgyeKnBJg: Downloading web safari player API JSON
[info] ixcgyeKnBJg: Downloading 1 format(s): 251
[download] Destination: temp
[download] 100% of   58.45MiB in 00:00:31 at 1.86MiB/s   
[ExtractAudio] Destination: temp.mp3
Deleting original file temp (pass -k to keep)
>>>>> Download finished in temp.mp3

TRANS: temp.mp3 with turbo model
--------------------


/home/nando/anaconda3/envs/ds_cont/lib/python3.12/site-packages/torch/cuda/__init__.py:283: UserWarning: 
    Found GPU0 NVIDIA GeForce MX230 which is of cuda capability 6.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  warnings.warn(
/home/nando/anaconda3/envs/ds_cont/lib/python3.12/site-packages/torch/cuda/__init__.py:304: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  warnings.warn(matched_cuda_warn.format(matched_arches))
/home/nando/anaconda3/envs/ds_cont/lib/python3.12/site-packages/torch/cuda/__init__.py:326: UserWarning: 
NVIDIA GeForce MX230 with CUDA capability sm_61 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the NVIDIA GeForce MX230 GPU with PyTorch, please check the

RuntimeError: Error(s) in loading state_dict for Whisper:
	While copying the parameter named "encoder.blocks.0.attn.query.weight", whose dimensions in the model are torch.Size([1280, 1280]) and whose dimensions in the checkpoint are torch.Size([1280, 1280]), an exception occurred : ("CUDA error: no kernel image is available for execution on the device\nSearch for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.\nCUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.\nFor debugging consider passing CUDA_LAUNCH_BLOCKING=1\nCompile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.\n",).
	While copying the parameter named "encoder.blocks.0.attn.key.weight", whose dimensions in the model are torch.Size([1280, 1280]) and whose dimensions in the checkpoint are torch.Size([1280, 1280]), an exception occurred : ("CUDA error: no kernel image